# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sandesh30-cloud/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd, numpy as np, os
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata
from scipy.stats import spearmanr

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
print(f'Loaded {len(raw)} rows, {len(raw.columns)} columns for March 2026.')

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gsc = raw[raw['gsc_data_available'] == True].copy()
print(f'{len(gsc)} of {len(raw)} rows have gsc_data_available == True.')

monthly = (gsc.groupby(['client_hash_id','content_hash_id'])
              .agg(impressions=('gsc_impressions','sum'),
                   clicks=('gsc_clicks','sum'),
                   avg_position=('gsc_avg_position', lambda s: s[s > 0].mean()),
                   days_with_position=('gsc_avg_position', lambda s: (s > 0).sum()))
              .reset_index())
monthly = monthly[(monthly['impressions'] > 0) & (monthly['avg_position'].notna())].copy()
monthly['ctr'] = monthly['clicks'] / monthly['impressions']
print(f'{len(monthly)} content items with usable position + impressions this month.')
monthly.head()

Signal 1 — CTR vs position (flag-linked: behind the CTR-fix flag)

In [ ]:
position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
monthly['position_bucket'] = pd.cut(monthly['avg_position'], bins=position_bins, labels=position_labels)

signal1_table = monthly.groupby('position_bucket', observed=True).agg(
    n=('ctr','size'), mean_ctr=('ctr','mean'), median_ctr=('ctr','median')
).reset_index()
print('--- Signal 1: CTR by position bucket ---')
print(signal1_table.to_string(index=False))

corr1, pval1 = spearmanr(monthly['avg_position'], monthly['ctr'])
if pval1 >= 0.05:
    verdict1 = 'FALSE'
elif corr1 < 0:
    verdict1 = 'CONFIRMED'
else:
    verdict1 = 'OPPOSITE'
print(f"\nSpearman correlation(position, ctr) = {corr1:.3f}, p = {pval1:.4f}")
print(f'SIGNAL 1 VERDICT: {verdict1}')

Signal 2 — content volume (flag-linked: behind quick-win)

In [ ]:
expected_ctr_map = signal1_table.set_index('position_bucket')['mean_ctr'].to_dict()

# FIX: position_bucket is a Categorical dtype (from pd.cut). Mapping it directly can leave
# the result as categorical too, which can't be subtracted with numpy ops (the TypeError above).
# Cast both sides to str for the lookup, then force the mapped result to float.
expected_ctr_map_str = {str(k): v for k, v in expected_ctr_map.items()}
monthly['expected_ctr'] = monthly['position_bucket'].astype(str).map(expected_ctr_map_str).astype(float)

monthly['ctr_shortfall'] = (monthly['expected_ctr'] - monthly['ctr']).clip(lower=0)
monthly['extra_clicks_opportunity'] = monthly['impressions'] * monthly['ctr_shortfall']

volume_bins = monthly['impressions'].quantile([0, .25, .5, .75, 1.0]).values
volume_bins[0] = -1
volume_labels = ['Q1_low','Q2','Q3','Q4_high']
monthly['volume_bucket'] = pd.cut(monthly['impressions'], bins=volume_bins, labels=volume_labels, duplicates='drop')

signal2_table = monthly.groupby('volume_bucket', observed=True).agg(
    n=('extra_clicks_opportunity','size'),
    mean_opportunity=('extra_clicks_opportunity','mean'),
    total_opportunity=('extra_clicks_opportunity','sum')
).reset_index()
print('--- Signal 2: click opportunity by volume bucket ---')
print(signal2_table.to_string(index=False))

corr2, pval2 = spearmanr(monthly['impressions'], monthly['extra_clicks_opportunity'])
if pval2 >= 0.05:
    verdict2 = 'FALSE'
elif corr2 > 0:
    verdict2 = 'CONFIRMED'
else:
    verdict2 = 'OPPOSITE'
print(f"\nSpearman correlation(impressions, opportunity) = {corr2:.3f}, p = {pval2:.4f}")
print(f'SIGNAL 2 VERDICT: {verdict2}')

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
MIN_IMPRESSIONS = 50
GOOD_POSITION_BUCKETS = ['1-3','4-6','7-10','11-20']

candidates = monthly[
    monthly['position_bucket'].isin(GOOD_POSITION_BUCKETS) &
    (monthly['impressions'] >= MIN_IMPRESSIONS) &
    (monthly['ctr_shortfall'] > 0)
].copy()

candidates['reason_code'] = 'CTR_FIX_OPPORTUNITY'
candidates['action'] = 'REWRITE_TITLE_META'
candidates['score'] = candidates['extra_clicks_opportunity']

ranked_queue = candidates.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = ['client_hash_id','content_hash_id','position_bucket','avg_position',
               'impressions','clicks','ctr','expected_ctr','ctr_shortfall',
               'score','reason_code','action']

os.makedirs('work/outputs', exist_ok=True)
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f'Wrote {len(ranked_queue)} ranked rows to work/outputs/baseline_action_score.csv')
ranked_queue[output_cols].head(20)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_queue.head(20)
for i, row in top20.iterrows():
    days_covered = monthly.loc[monthly['content_hash_id'] == row['content_hash_id'], 'days_with_position'].iloc[0]
    confidence = 'high confidence' if days_covered >= 15 else ('moderate confidence' if days_covered >= 7 else 'low confidence - thin sample')
    print(f"#{i+1} | action={row['action']} | reason_code={row['reason_code']} | score={row['score']:.1f}")
    print(f"  why: position bucket {row['position_bucket']} (avg_position={row['avg_position']:.1f}), "
          f"CTR {row['ctr']:.4f} vs bucket-expected {row['expected_ctr']:.4f}, "
          f"{row['impressions']:.0f} impressions across {days_covered} days with position data.")
    print(f"  confidence: {confidence}")
    print(f"  would be wrong if: the shortfall is a measurement artifact (e.g. a mid-month tracking gap) "
          f"rather than a real snippet/title problem, or if {days_covered} days is too thin to trust the CTR estimate.")
    print()

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Weak picks: thin coverage despite passing the impression floor ---
weak = top20[top20['content_hash_id'].isin(
    monthly.loc[monthly['days_with_position'] < 10, 'content_hash_id']
)]
if weak.empty:
    print('No top-20 picks rely on fewer than 10 days of position data this month - no weak picks by this check.')
else:
    print(f'{len(weak)} of the top 20 rely on fewer than 10 days of position data this month:')
    print(weak[['content_hash_id','score','impressions']].to_string(index=False))

# --- Leakage check ---
print('\n--- Leakage check ---')
cols_used_in_score = ['gsc_impressions','gsc_clicks','gsc_avg_position','gsc_data_available']
flag_like_cols = [c for c in raw.columns if any(k in c.lower() for k in
                  ['flag','trend','label','is_declining','is_flagged'])]
print(f"Columns loaded this notebook: {list(raw.columns)}")
print(f"Any product-flag / trend / label-style columns present: {flag_like_cols if flag_like_cols else 'none'}")
print(f"Score built only from: {cols_used_in_score}")

min_date, max_date = raw['report_date'].min(), raw['report_date'].max()
print(f"Date range actually loaded: {min_date.date()} to {max_date.date()}")
assert max_date < pd.Timestamp('2026-04-01'), 'Data extends past March 2026 - future window leaked in!'
print('Confirmed: no columns beyond raw GSC metrics used, and no dates beyond March 2026 loaded.')

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.